In [1]:
from sedona.spark import SedonaContext
import os
import time

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder().\
    config("spark.executor.memory", "12G").\
    config("spark.driver.memory", "16G").\
    config("sedona.join.autoBroadcastJoinThreshold", "-1").\
    config("spark.hadoop.fs.s3a.multipart.size", "24M").\
    config("spark.hadoop.fs.s3a.threads.max", "64").\
    config("spark.hadoop.fs.s3a.connection.maximum", "1000")

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/24 21:32:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/24 21:32:29 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/08/24 21:32:29 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/08/24 21:32:29 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/08/24 21:32:29 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/08/24 21:32:29 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/08/24 21:32:29 WARN SimpleFunctionRegistry: The function st_envelop

# Filter early

In [3]:
smaller_area = "POLYGON ((-121.861979 37.303503, -121.861979 37.428148, -121.988454 37.428148, -121.988454 37.303503, -121.861979 37.303503))"

In [4]:
buildings = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/optimizations/partitioned_geoparquet")

buildings.createOrReplaceTempView("buildings")

h3_cells = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/optimizations/h3_cells_california_parquet")

h3_cells.createOrReplaceTempView("h3_cells")

In [13]:
# filtering to smaller region at the end

In [ ]:
start = time.time()

result_not_optimized = sedona.sql(
f"""
WITH spatial_join AS (
    SELECT 
        b.geometry,
        h.geometry AS h3_geom,
        h.h3_id,
        b.subtype
    FROM buildings AS b
    JOIN h3_cells AS h ON ST_Intersects(b.geometry, h.geometry)
), aggregated AS (
    SELECT
        h3_id,
        subtype,
        FIRST(h3_geom) AS geometry,
        COUNT(*) AS count
    FROM spatial_join
    GROUP BY h3_id, subtype
)
SELECT 
    * 
FROM aggregated
WHERE ST_Contains(ST_GeomFromText('{smaller_area}'), geometry)
"""
    
)

print(result_not_optimized.count())
print(time.time() - start)

In [22]:
# filtering to region early

In [ ]:
start = time.time()

result_optimized = sedona.sql(
f"""
WITH h3_filtered AS (
    SELECT 
        * 
    FROM h3_cells
    WHERE ST_Contains(ST_GeomFromText('{smaller_area}'), geometry)
), buildings_filtered AS (
    SELECT 
        * 
    FROM buildings
    WHERE ST_Intersects(ST_GeomFromText('{smaller_area}'), geometry)
), spatial_join AS (
    SELECT 
        b.geometry,
        h.geometry AS h3_geom,
        h.h3_id,
        b.subtype
    FROM buildings_filtered AS b
    JOIN h3_filtered AS h ON ST_Intersects(b.geometry, h.geometry)
)
    SELECT
        h3_id,
        subtype,
        FIRST(h3_geom) AS geometry,
        COUNT(*) AS count
    FROM spatial_join
    GROUP BY h3_id, subtype
"""    
)

print(result_optimized.count())
print(time.time() - start)

[Stage 10:==============>                                         (8 + 11) / 32]

# Avoid Spheroid Distance in Joins

In [26]:
# distance join with transformation

In [28]:
start = time.time()

sedona.sql(
"""
WITH transformed AS (
    SELECT
        id,
        ST_Transform(geometry, 'EPSG:4326', 'EPSG:2229') AS geometry
    FROM buildings
)
SELECT 
    b1.*,
    b2.*
FROM transformed AS b1
JOIN transformed AS b2 ON ST_DWithin(b1.geometry, b2.geometry, 500)
"""
).count()
print(time.time() - start)

[Stage 61:======================================================> (31 + 1) / 32]

425.30585980415344


In [27]:
# distance join using spheroid function

In [ ]:
start = time.time()

sedona.sql(
"""
SELECT 
    b1.*,
    b2.*
FROM buildings AS b1
JOIN buildings AS b2 ON ST_DWithin(b1.geometry, b2.geometry, 500, TRUE)
"""
).count()
print(time.time() - start)

# Cache DataFrame Reused DataFrame

In [5]:
import pyspark.sql.functions as f

smaller_area = "POLYGON ((-121.861979 37.303503, -121.861979 37.428148, -121.988454 37.428148, -121.988454 37.303503, -121.861979 37.303503))"

infrastructure = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/optimizations/infrastructure.geoparquet")

segment = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/optimizations/segment.geoparquet")

places = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/optimizations/places.geoparquet")

water = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/optimizations/water.geoparquet")

water.createOrReplaceTempView("water")

In [6]:
water_pretransformed = sedona.sql(
"""
SELECT 
    ST_Transform(geometry,  'EPSG:4326', 'EPSG:2229') AS geometry,
    subtype
FROM water
"""
)

buildings_spatially_filtered = sedona.sql(
f"""
SELECT 
    id,
    ST_Buffer(
        ST_Transform(
            geometry,
            'EPSG:4326',
            'EPSG:2229'
        ),
        500
    ) AS geometry
FROM buildings
WHERE ST_Intersects(geometry, ST_GeomFromText('{smaller_area}'))
"""
)

buildings_water = buildings_spatially_filtered.alias("b").\
    join(
        water_pretransformed.alias("w"),
        f.expr("ST_Intersects(w.geometry, b.geometry)")
    ).\
    selectExpr(
        "b.id",
        "b.geometry AS b_geometry",
        "w.subtype"
    ).\
    groupBy("id", "subtype").\
    agg(
        f.expr("COUNT(*) AS count"),
        f.expr("FIRST(b_geometry) AS geometry")
    )

In [33]:
most_common_water_type = buildings_water.\
    withColumn(
        "rank",
        f.expr("RANK() OVER (PARTITION BY id ORDER BY count)")
    )

In [36]:
# without cache

In [ ]:
start = time.time()

infrastructure.alias("i").join(
    most_common_water_type.alias("m"), f.expr("ST_DWithin(m.geometry, ST_Transform(i.geometry, 'EPSG:4326', 'EPSG:2229'), 500)")
).count()

places.alias("p").join(
    most_common_water_type.alias("m"), f.expr("ST_DWithin(m.geometry, ST_Transform(p.geometry, 'EPSG:4326', 'EPSG:2229'), 500)")
).count()

segment.alias("p").join(
    most_common_water_type.alias("m"), f.expr("ST_DWithin(m.geometry, ST_Transform(p.geometry, 'EPSG:4326', 'EPSG:2229'), 500)")
).count()

print(time.time() - start)

[Stage 126:===================================================>   (16 + 1) / 17]

119.56063914299011


In [37]:
# with cache

In [35]:
start = time.time()
most_common_water_type.cache()

infrastructure.alias("i").join(
    most_common_water_type.alias("m"),
    f.expr("ST_DWithin(m.geometry, ST_Transform(i.geometry, 'EPSG:4326', 'EPSG:2229'), 500)")
).count()

places.alias("p").join(
    most_common_water_type.alias("m"), f.expr("ST_DWithin(m.geometry, ST_Transform(p.geometry, 'EPSG:4326', 'EPSG:2229'), 500)")
).count()

segment.alias("p").join(
    most_common_water_type.alias("m"), f.expr("ST_DWithin(m.geometry, ST_Transform(p.geometry, 'EPSG:4326', 'EPSG:2229'), 500)")
).count()

print(time.time() - start)

[Stage 190:===================================================>   (14 + 1) / 15]

86.25660300254822


# Avoid Wide Operations

In [8]:
places_aggregated = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/optimizations/neighbors").\
    select("id", "geometry", "names", "neighbors")

places_aggregated.createOrReplaceTempView("places_neighbors")

In [47]:
number_of_repeats = 10

In [51]:
wide_times = []
for _ in range(number_of_repeats):

    start = time.time()
    
    sedona.sql(
    """
    SELECT
        id,
        ST_ConvexHull(
            ST_Union_Aggr(neighbor_geom)
        ) AS convex_hull
    FROM (
        SELECT 
            id,
            geometry,
            n.geometry AS neighbor_geom
        FROM places_neighbors
        LATERAL VIEW EXPLODE(neighbors) AS n
    ) AS r
    GROUP BY r.id
    
    """
    ).selectExpr("SUM(ST_Area(convex_hull))").show()

    wide_times.append(time.time() - start)

print(sum(wide_times)/len(wide_times))

+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|        3.274831548976583|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|        3.274831548976583|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|        3.274831548976583|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|        3.274831548976583|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|        3.274831548976583|
+-------------------------+



...

+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|        3.274831548976583|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|        3.274831548976583|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|        3.274831548976583|
+-------------------------+



[Stage 409:==================================================>    (11 + 1) / 12]

+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|        3.274831548976583|
+-------------------------+

16.116394090652467


In [53]:
narrow_times = []

for _ in range(number_of_repeats):

    start = time.time()

    sedona.sql(
    """
    SELECT 
        id,
        ST_ConvexHull(
            ST_Union(
                Transform(neighbors, n->n.geometry)
            )
        ) AS convex_hull
    FROM places_neighbors
    """
    ).selectExpr("SUM(ST_Area(convex_hull))").show()

    narrow_times.append(time.time() - start)

print(sum(narrow_times)/len(narrow_times))

+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|         3.27483154897657|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|         3.27483154897657|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|         3.27483154897657|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|         3.27483154897657|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|         3.27483154897657|
+-------------------------+



...

+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|         3.27483154897657|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|         3.27483154897657|
+-------------------------+



+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|         3.27483154897657|
+-------------------------+



[Stage 470:=========================================>             (12 + 4) / 16]

+-------------------------+
|sum(st_area(convex_hull))|
+-------------------------+
|         3.27483154897657|
+-------------------------+

5.660003900527954


# Use the Window Function Over  groupBy and join

In [9]:
observations = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/optimizations/observations")

observations.createOrReplaceTempView("observations")

number_of_repeats = 10

In [58]:
# window

In [ ]:
import time

window_times = []

for _ in range(number_of_repeats):
    start = time.time()
    sedona.sql(
    """
    SELECT 
        id,
        geometry,
        price,
        date
    FROM (
        SELECT 
            *,
            rank() OVER (PARTITION BY id ORDER BY date DESC) AS rank
        FROM observations
    ) AS r
    WHERE rank = 1
    """
    ).count()

    window_times.append(time.time() - start)

print(sum(window_times)/len(window_times))

[Stage 531:>                                                      (0 + 11) / 12]

26.147415375709535


In [59]:
# join + group by

In [57]:
import time

join_times = []

for _ in range(10):
    start = time.time()
    
    sedona.sql(
    """
    SELECT 
        o.*
    FROM (
        SELECT 
            id,
            max(date) AS date
        FROM observations
        GROUP BY id
    ) AS r
    JOIN observations AS o ON o.id = r.id AND o.date = r.date
    """
    ).count()

    
    join_times.append(time.time() - start)

print(sum(join_times)/len(join_times))

[Stage 659:=====================================================> (65 + 2) / 67]

33.88189730644226
